In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# imp libs
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from pathlib import Path
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix

In [ ]:
df_test  = pd.read_csv("test.csv")

df_train = pd.read_csv("train.csv")
sample = pd.read_csv("Sample.csv")

In [ ]:
df_test["is_train"] = 0
df_train["is_train"] = 1

In [ ]:
df = pd.concat([df_train, df_test], ignore_index=True) #merged it so we can perform essentail eda and fe on both datasets

## **Main Code start:**

### Let's start with feature engi and EDA on both train and test data


In [ ]:
numerical_objs = df_train.select_dtypes(include=['int64', 'float64']).columns
len(numerical_objs)
df.info()
display(df['race'].unique())
display(df['religion'].unique())
display(df['gender'].unique())
display(df['disability'].unique())
display(df['created_date'].unique())
df[~df['race'].isna()]
df['created_date'] = pd.to_datetime(df['created_date'], yearfirst=True)
# creating separate day, month, year
df['Day'] = pd.to_datetime(df['created_date'], yearfirst=True).dt.day.astype(int)
df['Month'] = pd.to_datetime(df['created_date'], yearfirst=True).dt.month.astype(int)
df['Year'] = pd.to_datetime(df['created_date'], yearfirst=True).dt.year.astype(int)
df
df['weekday'] = pd.to_datetime(df['created_date'], yearfirst=True).dt.weekday.astype(int)
df['weekend'] = df['weekday'].isin([5,6]).astype(int) #to fetch the weekends
df['created_hour'] = pd.to_datetime(df['created_date']).dt.hour
df.post_id.unique()

df.info()
pd.get_dummies(df['race'])
### Handling categorical features: 
#race, religion, gender : categorical to numerical
df.isna().sum()
# race, religion, gender : categorical to numerical
cols = ['race', 'religion', 'gender']
for i in cols:
    df[i+'_missing'] = df[i].isna().astype(int)
    df[i]= df[i].fillna("Missing").astype(str)



### Handling comments:

In [ ]:
# handing missing vals:
df[df['comment'].isna()]
df['comment'] = df['comment'].fillna('').astype(str)
df['comment_len'] = df['comment'].str.len()
# df['comment'] = df['comment'].str.lower()
df['word_cnt'] = df['comment'].str.split().str.len()
# pd.set_option('display.max_colwidth', None)
# df['comment'].head()

# df['comment']= df['comment'].str.join(" ")
# cleaning and removing noise: 
df['comment'] = df['comment'].str.replace(r'http\S+', '', regex=True)
df['comment'] = df['comment'].str.replace(r'\s+', ' ', regex=True)
df[:3]['comment']
df_train['comment']
#some text intensive features:
df['num_!'] = df['comment'].str.count('!')
df['num_ques'] = df['comment'].str.count(r'\?')
pd.set_option('display.max_colwidth', None)
df['comment'].head(3)
### Handling Emojis and Imp columns
# df.info()
df['emoji_counts'] = df['emoticon_1']+df['emoticon_2']+df['emoticon_3']
s = "She might be a bright spot for a party keou on Oahu dominated by greedy criminals or ethically challenged individuals."
s.lower()
### Handling Votes and performing FE on them: Engagement signls

# **Let's see what all can we do with it:**
# 1. Total votes (we can simply sum them)
# 2. ratio of upvotes to downvotes(get the ratio)

# *These are strong internal signal that'd be useful for prediction*
# df.info()
# df[['upvote', 'downvote']]
df['votes_ratio'] = df['upvote']/(df['upvote']+df['downvote']+ 1e-9) #learning: to avoid inf (0/0) add 1e-9 in denominator [it's basically adding a raelly tiny number 0.0000000001 to avoid 0/0 case]

#  balancing the magnitude : popularity magnitude or negative feedback magnitude using logx
df['upvote_log1'] = np.log1p(df['upvote'])
df['downvote_log1'] = np.log1p(df['downvote'])

#engagement scorea nd intensiryt
df['engagement_score'] = df['upvote'] - df['downvote']
df['emoticon_density'] = df['emoji_counts'] / (df['word_cnt'] + 1)
df.columns
# df['post_id'].unique()
# df.groupby('post_id').size().
post_counts = df[df['is_train']==1].groupby('post_id').size().rename('post_comments_count')
# rest not seen will just point to 0
df['post_comments_count'] = df['post_id'].map(post_counts).fillna(0).astype(int)
df[:5]
# df.drop(['created_date'], axis = 1, inplace=True)
# df.dtypes
df.duplicated(subset=['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'label', 'is_train', 'Day', 'Month', 'Year',
       'weekday', 'race_missing', 'religion_missing', 'gender_missing',
       'comment_len', 'word_cnt', 'emoji_counts', 'votes_ratio', 'upvote_log1',
       'downvote_log1']).sum() # the comment col is list, to avoid typeerror 
# missing flags 
display(df.describe().T)
display(df.info())
# outliers 
def iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr  = q3 - q1
    lower, upper = q1-1.5*iqr, q3+1.5*iqr 
    return lower, upper, (s<lower).sum(), (s>upper).sum()

print("the Extreme valeus are :")
for i in ['upvote', 'downvote', 'if_1', 'if_2']:
    l,u,below,above = iqr(df[i])
    print(f"{i}: lower={l:.2f}, upper={u:.2f}, below={below}, above={above}")

# df['upvote'].quantile(0.75)
### Splitting before Encoding and all
df['disability'] = df['disability'].astype(int) #0 is False and 1 is True

display(df['race'].unique(),
df['religion'].unique(),
df['gender'].unique(),
df['post_id'].unique())

In [ ]:
# i'd somehow ignored the if_1 and 2 so let's resolve it - it'll give some boost to f1_score

df["if_1_log"] = np.log1p(df["if_1"]) #there are outliers
df["if_2_log"] = np.log1p(df["if_2"])

df["if_interaction_log"] = df["if_1_log"] * df["if_2_log"]

In [ ]:
#sentiment score
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm.auto import tqdm

nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()
comments_list = df['comment'].astype(str).tolist()

print("Running lightning-fast VADER Sentiment Analysis...")
sentiment_scores = [sia.polarity_scores(text)['compound'] for text in tqdm(comments_list)]
df['sentiment_vader'] = sentiment_scores
print("Done..")

In [ ]:
final_df = df.copy(deep=True)
# removing redundent cols:
final_df.drop(['created_date', 'Day', 'Month', 'Year'], axis=1, inplace=True)
final_df.drop(['upvote','downvote'], axis=1, inplace=True) #a few more
final_df.dtypes
#Splitting the train and test
train_df = final_df[final_df['is_train'] == 1].copy(deep= True)
test_df = final_df[final_df['is_train'] == 0].copy(deep= True)
print(train_df.shape
, final_df.shape, df_train.shape) #just to verfiy the splitting wnet smoothly

print(test_df.shape
, final_df.shape, df_test.shape)
### After splitting train and test data: Real ML 
train_df.columns
x_ = train_df.drop(columns=['label','post_id'])
y_ = train_df['label'].astype(int).values
x_.shape, y_.shape
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    x_,
    y_,
    test_size=0.2,
    stratify=y_,
    random_state=42
)

In [ ]:
# test_df.columns
X_train.shape, y_train.shape, X_val.shape, y_val.shape




## Encoding categorical features:


In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True) #to avoid crash on unseen dataset
categories = ['race', 'religion', 'gender']

X_train_ohe = ohe.fit_transform(X_train[categories])
X_val_ohe = ohe.transform(X_val[categories])
X_test_ohe = ohe.transform(test_df[categories])
train_df.shape

In [ ]:
print("Adarsh".isupper())

In [ ]:
#@title Tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),
    min_df=3,
    lowercase=True
)

X_train_word = word_tfidf.fit_transform(X_train['comment'])
X_val_word   = word_tfidf.transform(X_val['comment'])
X_test_word  = word_tfidf.transform(test_df['comment'])

In [ ]:
char_tfidf = TfidfVectorizer(
    max_features=5000,
    analyzer='char',
    ngram_range=(3,5)
)

X_train_char = char_tfidf.fit_transform(X_train['comment'])
X_val_char   = char_tfidf.transform(X_val['comment'])
X_test_char  = char_tfidf.transform(test_df['comment'])

In [ ]:
from scipy.sparse import hstack
X_train_tfidf = hstack([X_train_word, X_train_char])
X_val_tfidf   = hstack([X_val_word, X_val_char])
X_test_tfidf  = hstack([X_test_word, X_test_char])
# X_train_text = hstack([X_train_word, X_train_char])
# X_val_text   = hstack([X_val_word, X_val_char])
# X_test_text  = hstack([X_test_word, X_test_char])

#adding the embeddings 
train_text_emb = np.load('/kaggle/input/datasets/kradarsh04/embeddings/train_embeddings.npy')
val_text_emb = np.load('/kaggle/input/datasets/kradarsh04/embeddings/val_embeddings.npy')
test_text_emb = np.load('/kaggle/input/datasets/kradarsh04/embeddings/test_embeddings.npy')
# join all the sparse matrices

from scipy.sparse import csr_matrix

X_train_emb_sparse = csr_matrix(train_text_emb)
X_val_emb_sparse   = csr_matrix(val_text_emb)
X_test_emb_sparse  = csr_matrix(test_text_emb)
# join

from scipy.sparse import hstack
#mergin the ohe and main dataset
numerical_col = ['emoticon_1', 'emoticon_2', 'emoticon_3', 'if_1', 
                'if_2','disability', 
                'weekday', 'weekend', 'created_hour', 'race_missing',
                'religion_missing', 'gender_missing', 'comment_len', 'word_cnt',
                'num_!', 'num_ques', 'emoji_counts', 'votes_ratio', 'upvote_log1',
                'downvote_log1', 'engagement_score', 'emoticon_density','if_interaction_log','sentiment_vader'] #excluded: ['post_id', 'comments']

#scalling the numericals 
from sklearn.preprocessing import StandardScaler, RobustScaler
scaler = StandardScaler()
Rscaler = RobustScaler() #changing this to see any improvs

X_train_num = Rscaler.fit_transform(X_train[numerical_col])
X_val_num   = Rscaler.transform(X_val[numerical_col])
X_test_num  = Rscaler.transform(test_df[numerical_col])
# final dataset
X_train_final = hstack([
    X_train_tfidf,
    X_train_emb_sparse,
    X_train_ohe,
    X_train_num
])

X_val_final = hstack([
    X_val_tfidf,
    X_val_emb_sparse,
    X_val_ohe,
    X_val_num
])

X_test_final = hstack([
    X_test_tfidf,
    X_test_emb_sparse,
    X_test_ohe,
    X_test_num
])

In [ ]:
# X_train_text = hstack([X_train_word, X_train_char])
# X_val_text   = hstack([X_val_word, X_val_char])
# X_test_text  = hstack([X_test_word, X_test_char])

In [ ]:
# from scipy.sparse import hstack
# #mergin the ohe and main dataset
# numerical_col = ['emoticon_1', 'emoticon_2', 'emoticon_3', 'if_1', 
#                 'if_2','disability', 
#                 'weekday', 'weekend', 'created_hour', 'race_missing',
#                 'religion_missing', 'gender_missing', 'comment_len', 'word_cnt',
#                 'num_!', 'num_ques', 'emoji_counts', 'votes_ratio', 'upvote_log1',
#                 'downvote_log1', 'engagement_score', 'emoticon_density',
#                 'post_comments_count'] #excluded: ['post_id', 'comments']

# #scalling the numericals 
# from sklearn.preprocessing import StandardScaler, RobustScaler
# scaler = StandardScaler()
# Rscaler = RobustScaler() #changing this to see any improvs

# X_train_num = Rscaler.fit_transform(X_train[numerical_col])
# X_val_num   = Rscaler.transform(X_val[numerical_col])
# X_test_num  = Rscaler.transform(test_df[numerical_col])

In [ ]:
# encoded_df = pd.DataFrame(X_train_ohe.toarray(), columns= ohe.get_feature_names_out(['race', 'religion', 'gender']))
# display(encoded_df.head()) # the encoded data
### Using TF-IDF for comments

# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf = TfidfVectorizer(max_features=20000, 
#                         ngram_range=(1,2), 
#                         min_df=5, 
#                         lowercase=True
# )

# X_train_text = tfidf.fit_transform(X_train['comment'])
# X_val_text = tfidf.transform(X_val['comment'])
# X_test_text = tfidf.transform(test_df['comment'])
# print(X_train_text.shape)
#prev: (198000, 20000)
# train_df.drop(['post_id'], axis=1, inplace=True) #will decide later

#Merging everything 
# from scipy.sparse import hstack

# X_train_final = hstack([X_train_text, X_train_ohe, X_train_num])
# X_val_final   = hstack([X_val_text, X_val_ohe, X_val_num])
# X_test_final  = hstack([X_test_text, X_test_ohe, X_test_num])

# from scipy.sparse import save_npz  #use load_npz for the same
# save_npz('X_train_final.npz', X_train_final)
# save_npz('X_val_final.npz', X_val_final)
# save_npz('X_test_final.npz', X_test_final)
# # y_train.('y_train.csv', index  = False)
# # y_val.to_csv('y_val.csv', index  = False)
# np.save('y_train', y_train)
# np.save('y_val', y_val)



In [ ]:
# X_train_split, X_val_split, y_train_split, y_val_split =  X_train_final, X_val_final,y_train,y_val

In [ ]:
import copy
X_train_split = copy.deepcopy(X_train_final)
X_val_split, y_train_split, y_val_split = copy.deepcopy(X_val_final),copy.deepcopy(y_train),copy.deepcopy(y_val)

In [ ]:
unique, counts = np.unique(y_val_split, return_counts=True)
print("\nClass distribution in Validation set:")
for label, count in zip(unique, counts):
    print(f"Label {label}: {count} rows")

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr = LogisticRegression(max_iter= 1000, class_weight='balanced', C=1.0, n_jobs=-1)

lr.fit(X_train_split, y_train_split)
y_pred = lr.predict(X_val_split)

print(classification_report(y_val_split, y_pred))
print("f1_sctrore: ", f1_score(y_val_split, y_pred, average='macro'))

In [ ]:
param_grid = {'C': [0.1, 1.0, 5.0]}
lr_base = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=500)

print("Starting Lightning-Fast Grid Search...")
grid_search = GridSearchCV(
    estimator=lr_base,
    param_grid=param_grid,
    scoring='f1_macro', 
    cv=3,               
    verbose=3
)
grid_search.fit(X_train_final, y_train_split)

In [ ]:
#@title: trying senstece transformer instead of tf-idf to improve context
# from sentence_transformers import SentenceTransformer
# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import loguniform

# import numpy as np

# import torch
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# train_text_emb = model.encode(
#     X_train["comment"].tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

# val_text_emb = model.encode(
#     X_val["comment"].tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

# test_text_emb = model.encode(
#     test_df["comment"].tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

In [ ]:
# np.save('train_embeddings.npy', train_text_emb)
# np.save('val_embeddings.npy', val_text_emb)
# np.save('test_embeddings.npy', test_text_emb)

### Hyperparameter tuning 

In [ ]:
X_train_final.shape

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight={
        0: 1,
        1: 1.2,
        2: 1,
        3: 3 
    },
    C=2.0,
    n_jobs=-1
)

lr.fit(X_train_final, y_train)
y_pred = lr.predict(X_val_final)

In [ ]:
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

param_grid = {
    "C": [1.0, 2.0, 3.0],
    "solver": ["saga"],
    "class_weight": [
        "balanced",
        {0:1, 1:1.2, 2:1, 3:3},
        {0:1, 1:1.2, 2:1, 3:4}
    ]
}

lr = LogisticRegression(
    max_iter=1000,
    solver="saga",
    C=2.0,
    class_weight={0:1, 1:1.2, 2:1, 3:3},
    n_jobs=-1
)

grid = GridSearchCV(
    lr,
    param_grid=param_grid,
    scoring=make_scorer(f1_score, average="macro"),
    cv=3,
    n_jobs=-1,
    verbose=2
)


In [ ]:
grid.fit(X_train_final, y_train)
print(grid.best_params_)
best_lr = grid.best_estimator_

y_pred = best_lr.predict(X_val_final)

In [ ]:
print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))

In [ ]:
import joblib

# 'best_lr' is the variable where you stored your best estimator
model_filename = 'logistic_regression_model.joblib'

# Save the model to the Kaggle working directory
joblib.dump(best_lr, model_filename)

print(f"Model saved successfully as {model_filename}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    C=2.0,
    n_jobs=-1
)

lr.fit(X_train_final, y_train)
y_pred = lr.predict(X_val_final)

print(classification_report(y_val, y_pred))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))

In [ ]:
param_distributions = {
    'C': loguniform(1e-2, 1e2), 
    'solver': ['lbfgs'], 
    'max_iter': [500] 
}
lr_base = LogisticRegression(class_weight='balanced', n_jobs=-1)

In [ ]:
random_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=param_distributions,
    n_iter=5,
    scoring='f1_macro',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1  
)

In [ ]:
random_search.fit(X_train_split, y_train_split)

In [ ]:
print(f"Best Macro F1 Score (Cross-Validated): {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")

In [ ]:
best_lr = random_search.best_estimator_
y_pred_tuned = best_lr.predict(X_val_final)
from sklearn.metrics import classification_report
print(classification_report(y_val_split, y_pred_tuned))

In [ ]:
# for final submission : same as we trained : TF-IDF + encoded + numerical
# all of which already done in: X_test_final

# test_preds = lr.predict(X_test_final)
# sample['label'] = test_preds
# sample.to_csv('submission.csv', index=False)